# DropoutIA — Exploration et nettoyage des données

Ce notebook fait deux choses :
1. **Explorer** les données brutes (comprendre ce qu'on a)
2. **Nettoyer** les données (les rendre exploitables par un modèle ML)

À chaque étape, lisez le commentaire avant d'exécuter la cellule.

## 1. Chargement des données brutes

In [44]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
import nettoyage

df = pd.read_excel("../data/raw/decrochage_19072026.xlsx")
print(df.shape)
df.head()

(20000, 13)


,codeEleve,dateNaisEleve,id_Genre,nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,cd_com,CD_ETAB,Resultat
0,F143085459,2008-06-10 00:00:00.000,2,2A33101110,NaN,1,10.978333,4,441,1,4410105,01269D,0
1,R145137256,2008-12-04 00:00:00.000,1,2A31101110,NaN,3,5.246667,4,441,1,4410103,01273H,1
2,M146081162,2008-07-09 00:00:00.000,2,2A33101110,1.0,1,11.666667,4,421,1,4210103,01101W,0
3,E148126731,2007-06-04 00:00:00.000,2,2A33101110,2.0,2,10.770000,4,421,1,4210103,01102X,0
4,L147057423,2008-03-19 00:00:00.000,2,2A33101110,2.0,2,10.591667,4,421,1,4210109,01112H,0


## 2. Premier coup d'œil : structure et types

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   codeEleve        20000 non-null  str    
 1   dateNaisEleve    20000 non-null  str    
 2   id_Genre         20000 non-null  int64  
 3   nefstat          20000 non-null  str    
 4   NbrAbsence       9635 non-null   float64
 5   NbrRedoublement  20000 non-null  int64  
 6   MoyenneNotes     20000 non-null  float64
 7   CD_REG           20000 non-null  int64  
 8   CD_PROV          20000 non-null  int64  
 9   CD_MIL           20000 non-null  int64  
 10  cd_com           20000 non-null  int64  
 11  CD_ETAB          20000 non-null  str    
 12  Resultat         20000 non-null  int64  
dtypes: float64(2), int64(7), str(4)
memory usage: 2.0 MB


In [46]:
df.describe(include='all')

,codeEleve,dateNaisEleve,id_Genre,nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,cd_com,CD_ETAB,Resultat
count,20000,20000,20000.00000,20000,9635.000000,20000.000000,20000.000000,20000.0,20000.000000,20000.0,2.000000e+04,20000,20000.000000
unique,20000,2115,NaN,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42,NaN
top,F143085459,2009-07-03 00:00:00.000,NaN,2A33101110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01274J,NaN
freq,1,33,NaN,6673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1068,NaN
mean,NaN,NaN,1.47630,NaN,2.193461,0.514550,12.311596,4.0,427.939000,1.0,4.279495e+06,NaN,0.063000
std,NaN,NaN,0.49945,NaN,4.612528,0.860189,3.185375,0.0,9.520232,0.0,9.520202e+04,NaN,0.242969
min,NaN,NaN,1.00000,NaN,0.000000,0.000000,0.000000,4.0,421.000000,1.0,4.210101e+06,NaN,0.000000
25%,NaN,NaN,1.00000,NaN,0.000000,0.000000,10.053333,4.0,421.000000,1.0,4.210103e+06,NaN,0.000000
50%,NaN,NaN,1.00000,NaN,0.000000,0.000000,12.023333,4.0,421.000000,1.0,4.210109e+06,NaN,0.000000
75%,NaN,NaN,2.00000,NaN,2.000000,1.000000,14.618750,4.0,441.000000,1.0,4.410103e+06,NaN,0.000000


## 3. Valeurs manquantes — combien, et où ?

In [47]:
df.isnull().sum().sort_values(ascending=False)

NbrAbsence         10365
codeEleve              0
dateNaisEleve          0
id_Genre               0
nefstat                0
NbrRedoublement        0
MoyenneNotes           0
CD_REG                 0
CD_PROV                0
CD_MIL                 0
cd_com                 0
CD_ETAB                0
Resultat               0
dtype: int64

## 4. Distribution de la variable cible

Vérifiez le déséquilibre des classes : combien d'élèves à risque (ou décrocheurs)
par rapport aux autres ? C'est une information clé pour la suite (Semaine 3).

In [48]:
# Adaptez le nom de la colonne cible selon votre fichier
# (id_niveauRisqueInit pour le risque, Resultat pour le décrochage)
df['Resultat'].value_counts()

Resultat
0    18740
1     1260
Name: count, dtype: int64

In [49]:
# 1. Corrélations entre variables numériques
df.corr(numeric_only=True)

,id_Genre,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,cd_com,Resultat
id_Genre,1.000000,-0.118333,-0.175233,0.218312,NaN,0.004617,NaN,0.004617,-0.070105
NbrAbsence,-0.118333,1.000000,0.286996,-0.269141,NaN,-0.007764,NaN,-0.007768,0.326204
NbrRedoublement,-0.175233,0.286996,1.000000,-0.537325,NaN,0.069673,NaN,0.069674,0.403770
MoyenneNotes,0.218312,-0.269141,-0.537325,1.000000,NaN,-0.065514,NaN,-0.065515,-0.290277
CD_REG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CD_PROV,0.004617,-0.007764,0.069673,-0.065514,NaN,1.000000,NaN,1.000000,0.055271
CD_MIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cd_com,0.004617,-0.007768,0.069674,-0.065515,NaN,1.000000,NaN,1.000000,0.055272
Resultat,-0.070105,0.326204,0.403770,-0.290277,NaN,0.055271,NaN,0.055272,1.000000


## 5. Nettoyage des données

### 5.1 Gestion de `NbrAbsence` manquant



In [50]:
df = nettoyage.gerer_absences_manquantes(df, colonne="NbrAbsence")
df[["NbrAbsence", "NbrAbsence_manquant"]].head(10)

,NbrAbsence,NbrAbsence_manquant
0,0.0,1
1,0.0,1
2,1.0,0
3,2.0,0
4,2.0,0
5,0.0,0
6,0.0,1
7,0.0,0
8,7.0,0
9,0.0,0


### 5.2 Transformer la date de naissance en âge

In [51]:
df = nettoyage.calculer_age(df, colonne_date="dateNaisEleve")
df[["dateNaisEleve", "age"]].head()

,dateNaisEleve,age
0,2008-06-10,16
1,2008-12-04,15
2,2008-07-09,16
3,2007-06-04,17
4,2008-03-19,16


### 5.3 Encoder les colonnes catégorielles (texte -> nombre)

In [52]:
colonnes_categorielles = ["nefstat"]  
df = nettoyage.encoder_categorielles(df, colonnes_categorielles)
df.head()

,codeEleve,dateNaisEleve,id_Genre,nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,CD_REG,CD_PROV,CD_MIL,cd_com,CD_ETAB,Resultat,NbrAbsence_manquant,age
0,F143085459,2008-06-10,2,14,0.0,1,10.978333,4,441,1,4410105,01269D,0,1,16
1,R145137256,2008-12-04,1,0,0.0,3,5.246667,4,441,1,4410103,01273H,1,1,15
2,M146081162,2008-07-09,2,14,1.0,1,11.666667,4,421,1,4210103,01101W,0,0,16
3,E148126731,2007-06-04,2,14,2.0,2,10.770000,4,421,1,4210103,01102X,0,0,17
4,L147057423,2008-03-19,2,14,2.0,2,10.591667,4,421,1,4210109,01112H,0,0,16


### 5.4 Retirer les identifiants inutiles au modèle

In [53]:
df_clean = nettoyage.retirer_identifiants(df, ["codeEleve", "dateNaisEleve", "CD_ETAB", "CD_REG", "CD_MIL", "CD_PROV"])
df_clean.head()

,id_Genre,nefstat,NbrAbsence,NbrRedoublement,MoyenneNotes,cd_com,Resultat,NbrAbsence_manquant,age
0,2,14,0.0,1,10.978333,4410105,0,1,16
1,1,0,0.0,3,5.246667,4410103,1,1,15
2,2,14,1.0,1,11.666667,4210103,0,0,16
3,2,14,2.0,2,10.770000,4210103,0,0,17
4,2,14,2.0,2,10.591667,4210109,0,0,16


## 6. Sauvegarde des données nettoyées

In [54]:
df_clean.to_csv("../data/processed/donnees_clean.csv", index=False)
print("Fichier sauvegardé :", df_clean.shape)

Fichier sauvegardé : (20000, 9)


In [55]:
# 2. Différence entre décrocheurs et non-décrocheurs
df.groupby('Resultat')[['NbrAbsence', 'NbrRedoublement', 'MoyenneNotes', 'age']].mean()

,NbrAbsence,NbrRedoublement,MoyenneNotes,age
Resultat,,,,
0,0.899039,0.424493,12.551348,13.995624
1,3.401587,1.853968,8.745750,15.442063


In [56]:
df['nefstat'].dtype

dtype('int8')